<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">شکل درست، محور درست؟</h1><p style="text-align:right"><b>پرسش آزمایش:</b> چگونه خطای معنایی محور را حتی وقتی کد اجرا می‌شود پیدا کنیم؟</p><p style="text-align:right">پیش‌نیاز: <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-01/13-torch.html"><bdi dir="ltr">13-torch</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-01/14-index-device.html"><bdi dir="ltr">14-index-device</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-02/15-broadcast.html"><bdi dir="ltr">15-broadcast</bdi></a>، <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-02/16-reshape.html"><bdi dir="ltr">16-reshape</bdi></a></p><p style="text-align:right">این دفتر مستقل است و به اجرای دفتر دیگری نیاز ندارد. از بالا به پایین اجرا کنید؛ برای اجرای دوباره از ابتدا، <bdi dir="ltr">Kernel</bdi> را <bdi dir="ltr">Restart</bdi> و سپس <bdi dir="ltr">Run All</bdi> کنید. برای بازکردن لینک درس‌ها، سرور کتاب باید روی پورت ۸۰۰۰ اجرا شده باشد؛ راهنمای نصب در <a href="../../docs/NOTEBOOKS.md"><bdi dir="ltr">docs/NOTEBOOKS.md</bdi></a> است.</p>
</div>

In [ ]:
import sys
from pathlib import Path

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "mini_gpt" / "model.py").is_file()
             and (p / "data" / "sample.txt").is_file()), None)
if ROOT is None:
    raise RuntimeError("Keep notebooks inside the extracted project folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Python:", sys.executable)
print("Project:", ROOT)

import torch
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(17)

def inspect(name, value):
    print(name, "shape =", tuple(value.shape),
          "dtype =", value.dtype, "device =", value.device)


<div dir="rtl" style="text-align:right">
<p style="text-align:right">در این دفتر، شکل و معنای محورهای <bdi dir="ltr">Tensor</bdi> را با <bdi dir="ltr">PyTorch</bdi> بررسی می‌کنیم. <bdi dir="ltr">B</bdi> تعداد نمونه، <bdi dir="ltr">T</bdi> تعداد موقعیت و <bdi dir="ltr">C</bdi> تعداد ویژگی است؛ نام‌ها را کنار <bdi dir="ltr">Shape</bdi> بنویسید. در مثال نخست فقط سطر و ستون داریم. پیش از اجرا شکل ضرب عضو‌به‌عضو و ضرب ماتریسی را جدا پیش‌بینی کنید.</p>
</div>

In [ ]:
x = torch.tensor([[1.,2.,3.],[4.,5.,6.]])
inspect("x", x)
print("Elementwise:", x*x)
print("Matrix product:", x @ x.T)
print("reshape(3,2):", x.reshape(3,2))
print("transpose:", x.T)
assert not torch.equal(x.reshape(3,2), x.T)
assert torch.equal(x @ x.T, torch.tensor([[14.,32.],[32.,77.]]))


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">شکست آشکار و شکست خاموش</h2><p style="text-align:right">می‌خواهیم میانگین هر سطر را از همان سطر کم کنیم. چرا حذف <bdi dir="ltr">keepdim</bdi> در جدول دو‌در‌سه خطا می‌دهد، اما در جدول سه‌در‌سه ممکن است اجرا شود و غلط باشد؟</p>
</div>

In [ ]:
try:
    x - x.mean(dim=1)
except RuntimeError as error:
    print("Expected broadcasting failure:", error)
else:
    raise AssertionError("Expected shape mismatch")
square = torch.arange(1.,10.).reshape(3,3)
wrong = square - square.mean(dim=1)
correct = square - square.mean(dim=1, keepdim=True)
inspect("row means with keepdim", square.mean(1, keepdim=True))
print("Wrong row means:", wrong.mean(1))
print("Correct row means:", correct.mean(1))
torch.testing.assert_close(wrong.mean(1), torch.tensor([-3.,0.,3.]))
torch.testing.assert_close(correct.mean(1), torch.zeros(3))


<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">محور <bdi dir="ltr">Batch</bdi> را نگه داریم</h2><p style="text-align:right">قبل از اجرا شکل نتیجهٔ انتخاب موقعیت آخر و انتخاب یک نمونه را حدس بزنید. <bdi dir="ltr">T</bdi> را از ۴ به ۸ ببرید و فقط سطر تنظیمات را تغییر دهید.</p>
</div>

In [ ]:
B, T, C = 2, 4, 3
batch = torch.arange(B*T*C, dtype=torch.float32).reshape(B,T,C)
for name, value in [("batch",batch),("last position",batch[:,-1,:]),
                    ("one sample, keep B",batch[0:1]),("one sample, remove B",batch[0])]:
    inspect(name, value)
ids = torch.tensor([[0,1]], dtype=torch.long)
inspect("integer token IDs", ids)


<div dir="rtl" style="text-align:right">
<p style="text-align:right"><b>برداشت:</b> <bdi dir="ltr">Shape</bdi> درست شرط لازم است، نه کافی. با یک مثال عددی توضیح دهید کدام محور در <bdi dir="ltr">Broadcasting</bdi> هم‌تراز می‌شود؛ فقط به نبودن پیام خطا اعتماد نکنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">برگشت به کتاب</h2><p style="text-align:right">پیش‌بینی و نتیجهٔ اجرا را کنار هم بنویسید؛ اگر تفاوتی داشتند، دلیلش را توضیح دهید. سپس به <a target="_self" href="http://127.0.0.1:8000/part-03/chapter-02/16-reshape.html">درس مرتبط</a> برگردید و نتیجه را با توضیح آن مقایسه کنید.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">تمرین تکمیلی: میانگین هر بردار را کم کنید، سپس محورهای <bdi dir="ltr">Batch</bdi> و زمان را ادغام کنید</h2>
<p style="text-align:right"><bdi dir="ltr">Broadcasting</bdi> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">reshape</code> را در یک مسیر با نشانی قابل پیگیری ترکیب کنید. پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">T</code>، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">C</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">keepdim</code> در همین دفتر معرفی شده‌اند. مثال‌های قبلی این دفتر را نگه داشته‌ایم. اکنون دو تابع <bdi dir="ltr">TODO</bdi> را خودتان بنویسید؛ <bdi dir="ltr">INCOMPLETE</bdi> یعنی کار هنوز تمام نشده است.</p>
</div>

In [ ]:
from pathlib import Path
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر از هر بردار <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">C</code>تایی میانگین خودش را کم کنیم، ردیف <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">b*T+t</code> در خروجی صاف‌شده متعلق به کدام موقعیت است؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import math
import torch
torch.set_num_threads(1)
torch.manual_seed(17)
batch = torch.tensor([[[1.,2.,6.],[10.,20.,30.]],[[0.,3.,9.],[-2.,1.,4.]]])
print('B,T,C:',batch.shape)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right">تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">center_and_pack(x)</code> ابتدا میانگین هر بردارِ محور آخر را از همان بردار کم کند، سپس <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,C)</code> را به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B*T,C)</code> ببرد. ترتیب نمونه و زمان حفظ شود.</p>
</div>

In [ ]:
def center_and_pack(x):
    # TODO
    return None

In [ ]:
def test_exercise():
    result = center_and_pack(batch)
    if result is None: return False
    assert result.shape == (4,3)
    for b in range(2):
        for t in range(2):
            torch.testing.assert_close(result[b*2+t],batch[b,t]-batch[b,t].mean())
    x = torch.arange(40.).reshape(2,5,4)
    torch.testing.assert_close(center_and_pack(x).mean(-1),torch.zeros(10))
    return True

exercise_complete = test_exercise()
print("PASS" if exercise_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط مرکز یک موقعیت را با افزودن ۱۰۰ به همهٔ ویژگی‌هایش تغییر دهید. خروجی مرکز‌شدهٔ آن باید تقریباً ثابت بماند.</p>
</div>

In [ ]:
changed = batch.clone(); changed[1,0] += 100
print('centered change:',(changed-changed.mean(-1,keepdim=True))-(batch-batch.mean(-1,keepdim=True)))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right">نسخهٔ خراب پیش از صاف‌کردن، <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">B</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">T</code> را جابه‌جا کرده است. تابع <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">pack_batch_time(x)</code> فقط <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B,T,C)</code> را با ترتیب <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">b*T+t</code> به <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">(B*T,C)</code> ببرد.</p>
</div>

In [ ]:
wrong = batch.transpose(0,1).reshape(-1,3)
print('wrong row 1:',wrong[1],'expected source:',batch[0,1])

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def pack_batch_time(x):
    # TODO
    return None

In [ ]:
def test_repair():
    result = pack_batch_time(batch)
    if result is None: return False
    assert torch.equal(result[1],batch[0,1])
    assert torch.equal(result[2],batch[1,0])
    x = torch.arange(30).reshape(2,3,5)
    assert torch.equal(pack_batch_time(x)[4],x[1,1])
    return True

repair_complete = test_repair()
print("PASS" if repair_complete else "INCOMPLETE: complete the TODO first")

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">MiniGPT</code> در محاسبهٔ <bdi dir="ltr">Loss</bdi> دو محور <bdi dir="ltr">Batch</bdi> و <bdi dir="ltr">Time</bdi> را با همین ترتیب صاف می‌کند. درست‌بودن تعداد عناصر، ترتیب معنایی آن‌ها را تضمین نمی‌کند.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">کدام آزمون، جابه‌جایی خاموش نمونه و زمان را آشکار کرد؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-03/chapter-02/16-reshape.html">بازگشت به درس مرتبط</a> · <a target="_self" href="http://127.0.0.1:8000/answers/lab-03_tensor_shapes.html">فقط پس از تلاش: پاسخ مرجع تمرین تکمیلی</a></p></div>